# 2026 FIFA World Cup Prediction Engine — Phase 1 Submission Notebook

This notebook generates a complete baseline prediction using curated team-strength priors, Poisson score modeling, deterministic group ranking, and knockout slot resolution.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import re

group_fixtures = pd.read_csv('data/group_fixtures.csv')
knockout_slots = pd.read_csv('data/knockout_slots.csv')
print(group_fixtures.shape, knockout_slots.shape)

In [ ]:

TEAM_STRENGTH_PRIORS = {
    "Argentina": 2075, "France": 2070, "Spain": 2050, "England": 2025, "Brazil": 2015,
    "Portugal": 1995, "Netherlands": 1960, "Germany": 1945, "Uruguay": 1915, "Croatia": 1890,
    "Belgium": 1885, "Colombia": 1870, "Morocco": 1855, "USA": 1815, "Japan": 1810,
    "Switzerland": 1805, "Austria": 1795, "Ecuador": 1785, "Senegal": 1775, "Mexico": 1765,
    "Paraguay": 1740, "Iran": 1730, "South Korea": 1725, "Australia": 1715, "Norway": 1710,
    "Côte d'Ivoire": 1695, "Canada": 1685, "Scotland": 1680, "Egypt": 1675, "Algeria": 1665,
    "Qatar": 1605, "Tunisia": 1600, "South Africa": 1585, "Saudi Arabia": 1580, "Ghana": 1575,
    "Uzbekistan": 1565, "Panama": 1545, "New Zealand": 1535, "Jordan": 1510, "Haiti": 1495,
    "Curaçao": 1485, "Cabo Verde": 1480,
    "UEFA Playoff A": 1680, "UEFA Playoff B": 1660, "UEFA Playoff C": 1650, "UEFA Playoff D": 1640,
    "FIFA Playoff 1": 1540, "FIFA Playoff 2": 1540,
}

def rating(team):
    return TEAM_STRENGTH_PRIORS.get(team, 1600)

def expected_goals(home_rating, away_rating, base=1.35, scale=0.0035):
    diff = home_rating - away_rating
    return float(np.clip(base + scale * diff, 0.25, 3.75)), float(np.clip(base - scale * diff, 0.25, 3.75))

def score_matrix(home_rating, away_rating, max_goals=7):
    lh, la = expected_goals(home_rating, away_rating)
    goals = np.arange(max_goals + 1)
    mat = np.outer(poisson.pmf(goals, lh), poisson.pmf(goals, la))
    return mat / mat.sum()

def predict_score(home_team, away_team, knockout=False):
    hr, ar = rating(home_team), rating(away_team)
    mat = score_matrix(hr, ar)
    i, j = np.unravel_index(np.argmax(mat), mat.shape)
    home_win = np.tril(mat, -1).sum()
    away_win = np.triu(mat, 1).sum()
    if knockout and i == j:
        winner = 'home' if home_win >= away_win else 'away'
        penalties = True
    else:
        winner = 'home' if i > j else 'away' if j > i else 'draw'
        penalties = False
    rating_gap = abs(hr - ar)
    return {
        'predicted_home_goals': int(i),
        'predicted_away_goals': int(j),
        'corners': int(np.clip(round(9.2 + min(rating_gap, 350) / 350 * 0.8), 7, 12)),
        'yellow_cards': int(np.clip(round(3.6 + (1 if knockout else 0)), 2, 7)),
        'red_cards': 0,
        'winner': winner,
        'penalties': penalties,
    }


In [ ]:

group_predictions = group_fixtures.copy()
for col in ['predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','winning_team']:
    group_predictions[col] = None

for idx, row in group_predictions.iterrows():
    pred = predict_score(row.home_team, row.away_team, knockout=False)
    group_predictions.loc[idx, 'predicted_home_goals'] = pred['predicted_home_goals']
    group_predictions.loc[idx, 'predicted_away_goals'] = pred['predicted_away_goals']
    group_predictions.loc[idx, 'corners'] = pred['corners']
    group_predictions.loc[idx, 'yellow_cards'] = pred['yellow_cards']
    group_predictions.loc[idx, 'red_cards'] = pred['red_cards']
    group_predictions.loc[idx, 'winning_team'] = pred['winner']

def rank_group(g):
    teams = sorted(set(g.home_team) | set(g.away_team))
    table = {t: {'team': t, 'group': g.group.iloc[0], 'points': 0, 'goals_for': 0, 'goals_against': 0} for t in teams}
    for _, m in g.iterrows():
        h, a = m.home_team, m.away_team
        hg, ag = int(m.predicted_home_goals), int(m.predicted_away_goals)
        table[h]['goals_for'] += hg; table[h]['goals_against'] += ag
        table[a]['goals_for'] += ag; table[a]['goals_against'] += hg
        if hg > ag: table[h]['points'] += 3
        elif ag > hg: table[a]['points'] += 3
        else:
            table[h]['points'] += 1; table[a]['points'] += 1
    df = pd.DataFrame(table.values())
    df['goal_diff'] = df.goals_for - df.goals_against
    df = df.sort_values(['points','goal_diff','goals_for','team'], ascending=[False,False,False,True]).reset_index(drop=True)
    df['rank'] = np.arange(1, len(df)+1)
    return df

standings = pd.concat([rank_group(g) for _, g in group_predictions.groupby('group')], ignore_index=True)
group_predictions.head()


In [ ]:

def build_slot_map(standings):
    slots, thirds = {}, []
    for group, g in standings.groupby('group'):
        ranked = g.sort_values('rank')
        slots[f'Winner Group {group}'] = ranked.iloc[0].team
        slots[f'Runner-up Group {group}'] = ranked.iloc[1].team
        thirds.append(ranked.iloc[2].to_dict())
    thirds = sorted(thirds, key=lambda x: (x['points'], x['goal_diff'], x['goals_for'], x['team']), reverse=True)[:8]
    return slots, thirds

def resolve_slot(slot, slot_map, available_thirds):
    if slot in slot_map:
        return slot_map[slot]
    m = re.match(r'Best 3rd \(Groups ([A-L/]+)\)', slot)
    if m:
        allowed = set(m.group(1).split('/'))
        for idx, third in enumerate(available_thirds):
            if third['group'] in allowed:
                return available_thirds.pop(idx)['team']
        return available_thirds.pop(0)['team']
    m = re.match(r'Winner Match (\d+)', slot)
    if m: return slot_map[f'Winner Match {m.group(1)}']
    m = re.match(r'Loser Match (\d+)', slot)
    if m: return slot_map[f'Loser Match {m.group(1)}']
    raise ValueError(slot)

knockout_predictions = knockout_slots.copy()
for col in ['predicted_home_team','predicted_away_team','predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','match_winner','penalties']:
    knockout_predictions[col] = None

slot_map, thirds = build_slot_map(standings)
available_thirds = [dict(x) for x in thirds]
for idx, row in knockout_predictions.sort_values('match_id').iterrows():
    home = resolve_slot(row.slot_home, slot_map, available_thirds)
    away = resolve_slot(row.slot_away, slot_map, available_thirds)
    pred = predict_score(home, away, knockout=True)
    knockout_predictions.loc[idx, 'predicted_home_team'] = home
    knockout_predictions.loc[idx, 'predicted_away_team'] = away
    knockout_predictions.loc[idx, 'predicted_home_goals'] = pred['predicted_home_goals']
    knockout_predictions.loc[idx, 'predicted_away_goals'] = pred['predicted_away_goals']
    knockout_predictions.loc[idx, 'corners'] = pred['corners']
    knockout_predictions.loc[idx, 'yellow_cards'] = pred['yellow_cards']
    knockout_predictions.loc[idx, 'red_cards'] = pred['red_cards']
    knockout_predictions.loc[idx, 'match_winner'] = pred['winner']
    knockout_predictions.loc[idx, 'penalties'] = pred['penalties']
    winner_team = home if pred['winner'] == 'home' else away
    loser_team = away if pred['winner'] == 'home' else home
    slot_map[f'Winner Match {int(row.match_id)}'] = winner_team
    slot_map[f'Loser Match {int(row.match_id)}'] = loser_team

knockout_predictions.tail()


In [ ]:
# Final validation: no empty prediction cells
assert group_predictions[['predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','winning_team']].notna().all().all()
assert knockout_predictions[['predicted_home_team','predicted_away_team','predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','match_winner','penalties']].notna().all().all()

print('Group predictions:', group_predictions.shape)
print('Knockout predictions:', knockout_predictions.shape)
print('Predicted final:')
knockout_predictions[knockout_predictions['match_id'] == 104][['predicted_home_team','predicted_away_team','predicted_home_goals','predicted_away_goals','match_winner','penalties']]

## Notes

This is the first complete baseline. It should be improved with historical results, dynamic Elo, odds calibration, player availability, and count models for corners/cards before final submission.